# NB7 — Safe Adaptation, Baselines, Stress Tests & Trust Score

Adjusted-plan implementation. **G = primary; I = exploratory/post-hoc; F = reference.** Offline synthetic evaluation only.

- Design/tune baseline thresholds using the 14 train/validation episodes with episode-level LOEO; score the four test episodes once, without tuning.
- Feature-set conclusions on the test episodes are labelled **already seen**.
- Ground-truth components are scoring-only, never correction/detector inputs.
- Separate pure `NORMAL`/`EVENT` unsafe corrections from `EVENT_DRIFT` event distortion.
- The exact paper Eq. 4 is not present in the available artifacts. The trust-score section explicitly records this as blocked rather than inventing a formula.
- Saved NB6 predictions cover `test` and `validation_crossfit` only. They do **not** by themselves establish 14-fold LOEO classifier predictions. LOEO below therefore evaluates the explicitly defined feature-based baselines, not NB6 classifier generalization.

## 1. Load and validate NB4/NB6 artifacts

In [3]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

_here = Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "data" / "processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run Jupyter from within the project tree; expected data/processed/.")
FEATURE_PATH = ROOT / "data/processed/feature_dataset_AF_v2.parquet"
NB6_OUT = ROOT / "outputs/notebook06_v2"
PRED_PATH = NB6_OUT / "nb6_v2_predictions.parquet"
CONFIG_PATH = NB6_OUT / "nb6_v2_config.json"
OUT = ROOT / "outputs/notebook07"
OUT.mkdir(parents=True, exist_ok=True)
for p in [FEATURE_PATH, PRED_PATH, CONFIG_PATH]:
    if not p.exists(): raise FileNotFoundError(f"Missing required input: {p}")
T = pd.read_parquet(FEATURE_PATH)
PRED = pd.read_parquet(PRED_PATH)
with open(CONFIG_PATH, encoding="utf-8") as f: NB6_CONFIG = json.load(f)
print("Root:", ROOT, "\nFeatures:", T.shape, "\nNB6 predictions:", PRED.shape)
print("Prediction parts:", PRED.part.value_counts(dropna=False).to_dict())


Root: C:\AIoT_Research 
Features: (11826, 53) 
NB6 predictions: (25536, 16)
Prediction parts: {'test': 14808, 'validation_crossfit': 10728}


## 2. Join saved decisions and enforce episode-level split discipline

In [5]:
def require(df, cols, label):
    missing = sorted(set(cols)-set(df.columns))
    if missing: raise ValueError(f"{label} missing columns: {missing}")
require(T, ["window_id","episode_id","split","label","variant"], "features")
require(PRED, ["window_id","episode_id","variant","label","split","regime","set","part","pred","confidence","decision"], "NB6 predictions")
require(PRED, [f"p_{x}" for x in ["NORMAL","DRIFT","EVENT","EVENT_DRIFT"]], "NB6 probabilities")
# Check uniqueness BEFORE deduplication: silent duplicate collapse would conceal a data problem.
if T.window_id.duplicated().any():
    raise ValueError("Feature table has duplicate window_id; resolve upstream rather than silently deduplicating.")
optional = ["dev_mean_adj","peer_level_rel","agree_sign","gt_drift_component","gt_event_component","original_value","modified_value","t_mean","peer_median_level","window_pure","label_purity","peer_median_dev"]
feat_cols = ["window_id"] + [x for x in optional if x in T.columns]
D = PRED.merge(T[feat_cols], on="window_id", how="left", validate="many_to_one", suffixes=("", "_feature"))
D["evidence_status"] = np.where(D["set"].eq("I"), "already_seen_test__exploratory_post_hoc", "already_seen_test__comparison")
# The intended split has 14 train/validation episodes and four test episodes.
episodes = T[["episode_id","split"]].drop_duplicates()
print("Episode counts by split:", episodes.groupby("split").episode_id.nunique().to_dict())
print("Feature sets:", sorted(D["set"].dropna().unique()), "| regimes:", sorted(D["regime"].dropna().unique()))


Episode counts by split: {'test': 4, 'train': 10, 'validation': 4}
Feature sets: ['F', 'G', 'I'] | regimes: ['ALL', 'W']


## 3. Correction estimator and evaluation metrics

The estimator uses smoothed peer deviation (`dev_mean_adj`) and a bound learned from training rows only. Correction is reversible: save the raw value, estimated correction, corrected value, and bound. This is a **window-level proxy**; it must not be described as validated minute-level signal reconstruction.

In [7]:
if "dev_mean_adj" not in D or "t_mean" not in D:
    raise ValueError("Correction requires dev_mean_adj and t_mean; refusing to emit a misleading corrected-value proxy.")
train_feat = T[(T.split == "train") & T.dev_mean_adj.notna()]
if train_feat.empty: raise ValueError("No training rows available to derive correction bound.")
CORRECTION_BOUND = max(float(train_feat.dev_mean_adj.abs().quantile(.95)), 1e-6)
D["correction_bound"] = CORRECTION_BOUND
D["estimated_drift"] = D.dev_mean_adj.clip(-CORRECTION_BOUND, CORRECTION_BOUND)
D["correction_attempt"] = D.decision.eq("correct")
D["corrected_value_proxy"] = D.t_mean - np.where(D.correction_attempt, D.estimated_drift, 0.0)
D["raw_value_proxy"] = D.t_mean
print("Train-only correction bound:", CORRECTION_BOUND)
print("NB6 correction attempts:", int(D.correction_attempt.sum()))


Train-only correction bound: 3.7653087174650772
NB6 correction attempts: 3831


## 4. Baselines

Compare no correction, always-correct peer-median proxy, a fixed threshold, CUSUM, and Page-Hinkley. The threshold is chosen inside each LOEO fold to maximize DRIFT recall subject to at most 5% correction on pure NORMAL/EVENT windows. CUSUM and Page-Hinkley use fixed, documented reference settings. These are baselines, not optimized detector claims.

In [9]:
# Baseline utilities. Higher absolute deviation is evidence for drift; truth labels are scoring-only.
def score_binary(y_true, score):
    y = np.asarray(y_true, dtype=int); s = np.asarray(score, dtype=float)
    ok = np.isfinite(s)
    if ok.sum() == 0 or len(np.unique(y[ok])) < 2: return {"roc_auc": np.nan, "pr_auc": np.nan}
    return {"roc_auc": roc_auc_score(y[ok], s[ok]), "pr_auc": average_precision_score(y[ok], s[ok])}

def page_hinkley_flags(values, delta=0.005, threshold=0.5):
    x = np.asarray(values, dtype=float); out = np.zeros(len(x), dtype=bool)
    x = np.where(np.isfinite(x), x, 0.0)
    mean = 0.; cum = 0.; min_cum = 0.
    for i, v in enumerate(x):
        mean += (v-mean)/(i+1); cum += v-mean-delta; min_cum = min(min_cum, cum)
        out[i] = (cum-min_cum) > threshold
    return out

def cusum_flags(values, k=0.05, h=0.5):
    x = np.asarray(values, dtype=float); out = np.zeros(len(x), dtype=bool); pos=neg=0.
    for i,v in enumerate(np.where(np.isfinite(x), x, 0.0)):
        pos=max(0.,pos+v-k); neg=min(0.,neg+v+k); out[i]=(pos>h or neg < -h)
    return out

# Peer median: use peer_level_rel if available, else dev_mean_adj as the documented proxy.
PEER_COL = "peer_level_rel" if "peer_level_rel" in D.columns else "dev_mean_adj"
D["baseline_no_correction"] = D["t_mean"]
D["baseline_peer_median_estimate"] = D[PEER_COL].astype(float).clip(-CORRECTION_BOUND, CORRECTION_BOUND)
# Fixed threshold is set in LOEO below; placeholder values are filled fold-wise for train/validation.
D["baseline_fixed_threshold_correct"] = False
D["baseline_cusum_correct"] = False
D["baseline_page_hinkley_correct"] = False


## 5. Leave-one-episode-out design on the 14 train/validation episodes

LOEO is computed from the feature table so each held-out episode is excluded from threshold selection. It does not claim to be LOEO evaluation of the NB6 classifier: the saved predictions do not contain predictions from all 14 held-out-episode folds.

In [11]:
# Feature-based LOEO: each held-out episode is excluded from threshold selection.
# Only train/validation rows enter design. Label/ground truth are used for threshold objective and scoring,
# never as detector inputs at prediction time.
required = ["episode_id", "split", "label", "dev_mean_adj"]
missing = [c for c in required if c not in T.columns]
if missing:
    raise ValueError(f"Cannot run feature-baseline LOEO; missing columns: {missing}")
DEV = T[T.split.isin(["train", "validation"])].copy()
DEV = DEV[DEV.label.isin(["NORMAL", "DRIFT", "EVENT", "EVENT_DRIFT"])].copy()
DEV["dev_abs"] = DEV.dev_mean_adj.abs()

def choose_threshold(train_fold):
    x=train_fold.dev_abs.to_numpy(float); y=train_fold.label.to_numpy()
    valid=np.isfinite(x)
    if not valid.any(): return np.inf
    candidates=np.unique(np.quantile(x[valid], np.linspace(.05,.99,40)))
    pure=np.isin(y,["NORMAL","EVENT"]); drift=(y=="DRIFT")
    best=(-1.,float(np.max(candidates)))
    for th in candidates:
        pred=valid & (x>=th)
        unsafe=float(pred[pure].mean()) if pure.any() else 0.
        recall=float(pred[drift].mean()) if drift.any() else 0.
        if unsafe<=.05 and recall>best[0]: best=(recall,float(th))
    return best[1]

def baseline_flags(frame, threshold):
    x=frame.dev_mean_adj.to_numpy(float)
    finite=np.isfinite(x)
    out={"no_correction":np.zeros(len(frame),bool),
         "always_correct_peer_median":finite.copy(),
         "fixed_threshold":finite & (np.abs(x)>=threshold),
         "CUSUM":cusum_flags(x,k=.05,h=.5),
         "Page_Hinkley":page_hinkley_flags(x,delta=.005,threshold=.5)}
    # Missing peer evidence must not trigger a correction.
    out["CUSUM"] &= finite; out["Page_Hinkley"] &= finite
    return out

fold_rows=[]
for held in sorted(DEV.episode_id.dropna().unique()):
    tr=DEV[DEV.episode_id.ne(held)]; va=DEV[DEV.episode_id.eq(held)]
    th=choose_threshold(tr)
    for method, flags in baseline_flags(va,th).items():
        z=va.copy(); z["correct_flag"]=flags
        pure=z.label.isin(["NORMAL","EVENT"]); drift=z.label.eq("DRIFT")
        fold_rows.append({"held_episode":held,"method":method,"n":len(z),"threshold":th if method=="fixed_threshold" else np.nan,
          "pure_unsafe_correction_rate":float(z.loc[pure,"correct_flag"].mean()) if pure.any() else np.nan,
          "drift_correction_recall":float(z.loc[drift,"correct_flag"].mean()) if drift.any() else np.nan,
          "event_drift_correction_rate":float(z.loc[z.label.eq("EVENT_DRIFT"),"correct_flag"].mean()) if z.label.eq("EVENT_DRIFT").any() else np.nan})
LOEO=pd.DataFrame(fold_rows)
print("LOEO baseline rows:",len(LOEO),"| held-out episodes:",LOEO.held_episode.nunique())
print(LOEO.groupby("method")[["pure_unsafe_correction_rate","drift_correction_recall","event_drift_correction_rate"]].mean().round(4).to_string())
print("Scope: feature-based baseline LOEO only; not NB6 classifier LOEO.")

LOEO baseline rows: 70 | held-out episodes: 14
                            pure_unsafe_correction_rate  drift_correction_recall  event_drift_correction_rate
method                                                                                                       
CUSUM                                            0.9578                   0.9380                       0.9593
Page_Hinkley                                     0.5473                   0.6562                       0.9139
always_correct_peer_median                       0.9610                   0.9593                       0.9593
fixed_threshold                                  0.0508                   0.7267                       0.7165
no_correction                                    0.0000                   0.0000                       0.0000
Scope: feature-based baseline LOEO only; not NB6 classifier LOEO.


## 6. Stress tests at evaluation time

Recompute each feature-based detector's correction flags under peer dropout, a contaminated peer, and common-mode drift. Compare unsafe-correction rate and DRIFT recall with the unstressed case. NB6's saved decisions are also summarized separately as a frozen-policy reference; they are not presented as an adaptive response to stress.

In [13]:
def stress_deviation(df, stress, seed=7, dropout_fraction=.5, contamination_fraction=.25):
    x=df.dev_mean_adj.astype(float).to_numpy(copy=True); rng=np.random.default_rng(seed)
    if stress=="none": return x
    if stress=="peer_dropout":
        x[rng.random(len(x))<dropout_fraction]=np.nan
    elif stress=="contaminated_peer":
        mask=rng.random(len(x))<contamination_fraction
        x[mask]=x[mask]+max(CORRECTION_BOUND*.5,1e-6)
    elif stress=="common_mode_drift":
        x=x+CORRECTION_BOUND*.5
    else: raise ValueError(stress)
    return x

# Freeze one threshold using all 14 design episodes, then apply it unchanged to test stress cases.
if DEV.empty: raise ValueError("No train/validation rows available for stress-test threshold fitting.")
FINAL_THRESHOLD=choose_threshold(DEV)
stress_rows=[]
for stress in ["none","peer_dropout","contaminated_peer","common_mode_drift"]:
    for (regime,fs),g in D[D.part.eq("test")].groupby(["regime","set"]):
        x=stress_deviation(g,stress,seed=7)
        finite=np.isfinite(x)
        flags={"no_correction":np.zeros(len(g),bool),
               "always_correct_peer_median":finite.copy(),
               "fixed_threshold":finite & (np.abs(x)>=FINAL_THRESHOLD),
               "CUSUM":cusum_flags(x,k=.05,h=.5)&finite,
               "Page_Hinkley":page_hinkley_flags(x,delta=.005,threshold=.5)&finite}
        # Saved NB6 decisions are reported as frozen reference, not recomputed policy.
        flags["NB6_saved_frozen"]=g.decision.eq("correct").to_numpy()
        for method,attempt in flags.items():
            y=g.label.to_numpy(); pure=np.isin(y,["NORMAL","EVENT"]); drift=(y=="DRIFT")
            stress_rows.append({"stress":stress,"regime":regime,"set":fs,"part":"test","method":method,"n":len(g),
              "peer_evidence_missing_share":float((~finite).mean()),
              "pure_unsafe_correction_rate":float(attempt[pure].mean()) if pure.any() else np.nan,
              "drift_correction_rate":float(attempt[drift].mean()) if drift.any() else np.nan,
              "mean_abs_peer_deviation":float(np.nanmean(np.abs(x))) if finite.any() else np.nan,
              "evidence_status":"already_seen_test; stress diagnostic"})
STRESS=pd.DataFrame(stress_rows)
# Include delta from the same method/regime/set's unstressed performance.
base=STRESS[STRESS.stress.eq("none")][["regime","set","method","pure_unsafe_correction_rate","drift_correction_rate"]].rename(columns={"pure_unsafe_correction_rate":"base_unsafe_rate","drift_correction_rate":"base_drift_rate"})
STRESS=STRESS.merge(base,on=["regime","set","method"],how="left")
STRESS["unsafe_rate_delta"]=STRESS.pure_unsafe_correction_rate-STRESS.base_unsafe_rate
STRESS["drift_rate_delta"]=STRESS.drift_correction_rate-STRESS.base_drift_rate
print(STRESS.round(4).to_string(index=False))
print("NB6_saved_frozen holds decisions fixed; other methods recompute detector flags from perturbed evidence.")

           stress regime set part                     method    n  peer_evidence_missing_share  pure_unsafe_correction_rate  drift_correction_rate  mean_abs_peer_deviation                      evidence_status  base_unsafe_rate  base_drift_rate  unsafe_rate_delta  drift_rate_delta
             none    ALL   F test              no_correction 2468                       0.0000                       0.0000                 0.0000                   1.8011 already_seen_test; stress diagnostic            0.0000           0.0000             0.0000            0.0000
             none    ALL   F test always_correct_peer_median 2468                       0.0000                       1.0000                 1.0000                   1.8011 already_seen_test; stress diagnostic            1.0000           1.0000             0.0000            0.0000
             none    ALL   F test            fixed_threshold 2468                       0.0000                       0.0640                 0.8190           

## 7. Trust score (paper Eq. 4) versus classifier confidence and baselines

This section is blocked until the paper's exact Eq. 4 and variable definitions are supplied. The notebook must not invent a substitute. Once transcribed, evaluate the score against the unsafe-decision target and compare it with classifier confidence and baseline predictors using ROC-AUC and PR-AUC. Test rows are descriptive only and are not used to select score weights or thresholds.

In [15]:
def paper_eq4_trust_score(frame):
    # Required: transcribe the paper's exact Eq. 4 and variable definitions here.
    raise NotImplementedError("Trust-score evaluation is blocked until the exact paper Eq. 4 is entered.")

def trust_predictor_evaluation(frame, trust_score):
    z=frame.copy(); z["paper_eq4_trust_score"]=np.asarray(trust_score)
    z["unsafe_decision"]=(z.decision.eq("correct") & z.label.isin(["NORMAL","EVENT"])).astype(int)
    z["confidence_unsafe_score"]=1-z.confidence.astype(float)
    z["peer_deviation_unsafe_score"]=z.dev_mean_adj.abs().astype(float)
    predictors={"paper_Eq4_trust_score":1-z.paper_eq4_trust_score,
                "classifier_confidence":z.confidence_unsafe_score,
                "peer_deviation_baseline":z.peer_deviation_unsafe_score}
    rows=[]
    for (regime,fs),g in z.groupby(["regime","set"]):
        y=g.unsafe_decision.to_numpy()
        for name,s_all in predictors.items():
            m=score_binary(y,s_all.loc[g.index].to_numpy())
            rows.append({"regime":regime,"set":fs,"predictor":name,"n":len(g),"unsafe_count":int(y.sum()),**m,
                         "evidence_status":"already_seen_test; descriptive only"})
    return pd.DataFrame(rows)

# Intentionally not called until exact Eq. 4 is transcribed; no invented trust-score results.

## 8. Test evaluation: correction error, event distortion, and unsafe correction size

Metrics use the saved NB6 decisions and ground-truth components only for scoring. Event distortion is the signed event-component change induced by the applied correction; report its absolute magnitude. These are already-seen test diagnostics, not confirmatory feature-set evidence.

In [29]:
test = D[D.part.eq("test")].copy()
metrics = []

if {"gt_drift_component", "gt_event_component"}.issubset(test.columns):
    test["applied_correction"] = np.where(
        test.correction_attempt,
        test.estimated_drift,
        0.0
    )

    # Remaining drift component after subtracting the applied correction.
    test["residual_drift_after_correction"] = (
        test.gt_drift_component - test.applied_correction
    )

    # Event distortion induced by the correction.
    test["event_distortion"] = np.sign(
        test.applied_correction * test.gt_event_component
    ) * np.minimum(
        np.abs(test.applied_correction),
        np.abs(test.gt_event_component)
    )

    for (regime, fs), g in test.groupby(["regime", "set"]):
        for cls in ["NORMAL", "EVENT", "DRIFT", "EVENT_DRIFT"]:
            z = g[g.label.eq(cls)]
            if z.empty:
                continue

            is_drift_class = cls in ["DRIFT", "EVENT_DRIFT"]
            is_pure_class = cls in ["NORMAL", "EVENT"]
            attempted = z[z.correction_attempt]

            metrics.append({
                "regime": regime,
                "set": fs,
                "class": cls,
                "n": len(z),

                # Report remaining drift magnitude for drift-containing windows.
                "residual_drift_mae_after_correction": (
                    float(z.residual_drift_after_correction.abs().mean())
                    if is_drift_class else np.nan
                ),

                "mean_abs_event_distortion": float(
                    z.event_distortion.abs().mean()
                ),

                "unsafe_attempt_rate": (
                    float(z.correction_attempt.mean())
                    if is_pure_class else np.nan
                ),

                "unsafe_correction_size_mean": (
                    float(attempted.estimated_drift.abs().mean())
                    if is_pure_class and not attempted.empty
                    else (0.0 if is_pure_class else np.nan)
                ),

                "event_drift_correction_rate": (
                    float(z.correction_attempt.mean())
                    if cls == "EVENT_DRIFT" else np.nan
                ),

                "evidence_status": "already_seen_test; I exploratory"
            })

    CORRECTION_METRICS = pd.DataFrame(metrics)
    print(CORRECTION_METRICS.round(4).to_string(index=False))

else:
    CORRECTION_METRICS = pd.DataFrame()
    print(
        "Ground-truth drift/event component columns absent; "
        "physical residual/event-distortion metrics unavailable."
    )


regime set       class    n  residual_drift_mae_after_correction  mean_abs_event_distortion  unsafe_attempt_rate  unsafe_correction_size_mean  event_drift_correction_rate                  evidence_status
   ALL   F      NORMAL  324                                  NaN                     0.0000               0.0062                       0.1794                          NaN already_seen_test; I exploratory
   ALL   F       EVENT  536                                  NaN                     0.0003               0.0019                       0.1529                          NaN already_seen_test; I exploratory
   ALL   F       DRIFT  536                               1.5232                     0.0000                  NaN                          NaN                          NaN already_seen_test; I exploratory
   ALL   F EVENT_DRIFT 1072                               2.2882                     0.0330                  NaN                          NaN                       0.0121 already_seen_

## 9. Save audit artifacts and limitations
## One qualification matters: these conclusions assume estimated_drift and gt_drift_component share the same scale and sign convention, and that subtracting the estimated correction models the intended correction operation. The new metric measures the residual component under that assumption, not independently validated real-sensor reconstruction.

In [19]:
D.to_parquet(OUT / "nb7_predictions_with_correction.parquet", index=False)
LOEO.to_csv(OUT / "nb7_loeo_baseline_results.csv", index=False)
STRESS.to_csv(OUT / "nb7_stress_test_results.csv", index=False)
CORRECTION_METRICS.to_csv(OUT / "nb7_correction_metrics.csv", index=False)
config={"source_predictions":str(PRED_PATH.relative_to(ROOT)),"source_features":str(FEATURE_PATH.relative_to(ROOT)),
        "primary_set":"G","exploratory_set":"I","reference_set":"F","correction_bound_train_abs_deviation_q95":CORRECTION_BOUND,
        "loeo_unit":"episode","loeo_scope":"feature-based baselines only; saved NB6 predictions do not establish 14-fold classifier LOEO",
        "design_episodes":"train + validation only (14 episodes)","test_episodes":"four; scored once; already seen for feature-set claims",
        "stress_tests":["peer_dropout","contaminated_peer","common_mode_drift"],
        "trust_score":"blocked until exact paper Eq. 4 is transcribed; no equation invented",
        "limitations":["synthetic injected scenarios","overlapping windows","four already-seen test episodes","I post-hoc exploratory",
                       "window-level correction proxy is not validated minute-level reconstruction",
                       "NB6 saved decisions are frozen during stress tests; baseline detectors are recomputed from perturbed evidence"]}
with open(OUT/"nb7_config.json","w",encoding="utf-8") as f: json.dump(config,f,indent=2,default=str)
print("Saved NB7 outputs:",OUT)
print("Interpret LOEO as feature-baseline evaluation only. Trust-score evaluation remains blocked pending exact paper Eq. 4.")

Saved NB7 outputs: C:\AIoT_Research\outputs\notebook07
Interpret LOEO as feature-baseline evaluation only. Trust-score evaluation remains blocked pending exact paper Eq. 4.
